# 🎙️ Voice Clone + Talking Avatar — SadTalker VERSION

## Why SadTalker instead of Wav2Lip?
| | Wav2Lip | SadTalker |
|---|---|---|
| Lip sync quality | ✅ Accurate | ✅ Accurate |
| Face realism | ❌ Blurry / smudgy | ✅ Sharp + natural |
| Head movement | ❌ None (static crop) | ✅ Natural head pose |
| Full face visible | ❌ Only mouth region | ✅ Full face + background |
| GFPGAN enhancement | ❌ | ✅ Built-in upscaler |

SadTalker generates audio-driven lip sync **and** natural 3D head motion from a single photo.  
The optional `--enhancer gfpgan` flag runs face restoration post-processing — eliminates the blurry artifact problem entirely.

## ⚠️ BEFORE RUNNING:
1. `Runtime → Change runtime type → T4 GPU → Save`
2. Run cells **one by one** — wait for each to finish

## Order:
```
CELL 1 → Check GPU
CELL 2 → Install SadTalker + dependencies
CELL 3 → Download SadTalker model weights
CELL 4 → Upload voice sample → generate cloned voice (F5-TTS)
CELL 5 → Upload face photo → generate avatar video (SadTalker)
CELL 6 → Generate new video with any custom text (reuse)
```

In [ ]:
# ============================================================
# CELL 1 — Check GPU
# ============================================================
import torch, sys
print(f'Python: {sys.version}')
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}' if torch.cuda.is_available() else '❌ No GPU — switch to T4 in Runtime settings')

In [ ]:
# ============================================================
# CELL 2 — Clone SadTalker + install dependencies
# ============================================================
import os

# Clone SadTalker
if not os.path.exists('/content/SadTalker'):
    !git clone https://github.com/OpenTalker/SadTalker.git /content/SadTalker -q
    print('✅ SadTalker cloned')
else:
    print('✅ SadTalker already cloned')

%cd /content/SadTalker

# Core dependencies
!pip install torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118 -q
!pip install face_alignment==1.3.5 -q
!pip install imageio==2.19.3 imageio-ffmpeg==0.4.7 -q
!pip install librosa==0.9.2 -q
!pip install numba -q
!pip install resampy==0.3.1 -q
!pip install pydub -q
!pip install scipy -q
!pip install kornia -q
!pip install tqdm -q
!pip install yacs -q
!pip install pyyaml -q

# GFPGAN face enhancer (removes blurry artifacts)
!pip install gfpgan -q

print('\n✅ All dependencies installed')

In [ ]:
# ============================================================
# CELL 3 — Download SadTalker model weights
# ============================================================
import os

os.makedirs('/content/SadTalker/checkpoints', exist_ok=True)
os.makedirs('/content/SadTalker/gfpgan/weights', exist_ok=True)

# ── SadTalker core weights ──────────────────────────────────
BASE = 'https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc'

WEIGHTS = [
    ('SadTalker_V0.0.2_256.safetensors', 'checkpoints'),
    ('SadTalker_V0.0.2_512.safetensors', 'checkpoints'),
    ('mapping_00109-model.pth.tar',       'checkpoints'),
    ('mapping_00229-model.pth.tar',       'checkpoints'),
    ('auido2exp_00300-model.pth.tar',     'checkpoints'),
    ('auido2pose_00140-model.pth.tar',    'checkpoints'),
    ('facevid2vid_00189-model.pth.tar',   'checkpoints'),
    ('shape_predictor_68_face_landmarks.dat', 'checkpoints'),
]

for fname, folder in WEIGHTS:
    dest = f'/content/SadTalker/{folder}/{fname}'
    if not os.path.exists(dest) or os.path.getsize(dest) < 1000:
        print(f'Downloading {fname}...')
        !wget -q --show-progress -O "{dest}" "{BASE}/{fname}"
    else:
        print(f'✅ {fname} already downloaded')

# ── GFPGAN weights (face enhancer) ─────────────────────────
GFPGAN_WEIGHTS = [
    ('https://github.com/xinntao/facexlib/releases/download/v0.1.0/detection_Resnet50_Final.pth',
     '/content/SadTalker/gfpgan/weights/detection_Resnet50_Final.pth'),
    ('https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth',
     '/content/SadTalker/gfpgan/weights/GFPGANv1.4.pth'),
    ('https://github.com/xinntao/facexlib/releases/download/v0.2.2/parsing_parsenet.pth',
     '/content/SadTalker/gfpgan/weights/parsing_parsenet.pth'),
]

for url, dest in GFPGAN_WEIGHTS:
    if not os.path.exists(dest) or os.path.getsize(dest) < 1000:
        fname = os.path.basename(dest)
        print(f'Downloading {fname}...')
        !wget -q --show-progress -O "{dest}" "{url}"
    else:
        print(f'✅ {os.path.basename(dest)} already downloaded')

print('\n✅ All model weights ready')

In [ ]:
# ============================================================
# CELL 4 — Install F5-TTS + generate cloned voice
# ============================================================
!pip install f5-tts -q

import os, shutil, subprocess
from google.colab import files

# Upload voice sample
print('Upload your voice sample (WAV, 6-30 sec):')
uploaded = files.upload()
original = list(uploaded.keys())[0]
VOICE_SAMPLE = '/content/voice_sample.wav'
shutil.copy(original, VOICE_SAMPLE)
print(f'✅ Voice sample saved: {VOICE_SAMPLE}')

# ── CHANGE THIS TEXT ────────────────────────────────────────
TEXT = "Artificial Intelligence is a transformative branch of computer science dedicated to creating systems capable of performing tasks that typically require human intelligence, such as reasoning, learning, problem-solving, and perception."

CLONED_AUDIO = '/content/cloned_voice.wav'

print(f'\nGenerating voice for: "{TEXT[:60]}..."')
result = subprocess.run([
    'f5-tts_infer-cli',
    '--model',       'F5TTS_v1_Base',
    '--ref_audio',   VOICE_SAMPLE,
    '--ref_text',    '',
    '--gen_text',    TEXT,
    '--output_dir',  '/content/',
    '--output_file', 'cloned_voice.wav',
    '--remove_silence',
], capture_output=True, text=True)

# F5-TTS sometimes appends a timestamp — find the file
if not os.path.exists(CLONED_AUDIO):
    wavs = sorted([f for f in os.listdir('/content') if f.endswith('.wav')])
    if wavs:
        shutil.copy(f'/content/{wavs[-1]}', CLONED_AUDIO)

if os.path.exists(CLONED_AUDIO):
    size_kb = os.path.getsize(CLONED_AUDIO) / 1024
    print(f'✅ Voice generated: {CLONED_AUDIO} ({size_kb:.0f} KB)')
    files.download(CLONED_AUDIO)
    print('✅ Downloaded — listen to verify before running Cell 5')
else:
    print('❌ Voice generation failed')
    print(result.stderr[-500:])

In [ ]:
# ============================================================
# CELL 5 — Upload face photo → generate realistic avatar (SadTalker)
# ============================================================
import os, shutil
from google.colab import files

# Verify audio exists
CLONED_AUDIO = '/content/cloned_voice.wav'
if not os.path.exists(CLONED_AUDIO):
    raise FileNotFoundError('❌ Run Cell 4 first to generate the voice')
print(f'✅ Audio: {CLONED_AUDIO}')

# Upload face photo
print('\nUpload face photo (JPG/PNG, single face, front-facing):')
uploaded = files.upload()
original = list(uploaded.keys())[0]
ext = os.path.splitext(original)[1].lower()
FACE_PHOTO = f'/content/face{ext}'
shutil.copy(original, FACE_PHOTO)
print(f'✅ Photo saved: {FACE_PHOTO}')

OUTPUT_DIR = '/content/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Run SadTalker ───────────────────────────────────────────
# Key flags:
#   --still          → minimal head movement (better for meetings)
#   --preprocess full → keep full image with background (not cropped)
#   --enhancer gfpgan → GFPGAN face restoration — eliminates blurry artifacts
#   --size 256       → use 256 model (faster); change to 512 for higher res

print('\nRunning SadTalker lip sync + face enhancement...')
print('(Takes 3-6 min on T4 GPU with GFPGAN enhancement)')

%cd /content/SadTalker
!python inference.py \
    --driven_audio  "{CLONED_AUDIO}" \
    --source_image  "{FACE_PHOTO}" \
    --result_dir    "{OUTPUT_DIR}" \
    --still \
    --preprocess    full \
    --enhancer      gfpgan \
    --size          256

# Find output video (SadTalker names it with a timestamp)
FINAL_VIDEO = None
for f in sorted(os.listdir(OUTPUT_DIR), reverse=True):
    if f.endswith('.mp4'):
        FINAL_VIDEO = os.path.join(OUTPUT_DIR, f)
        break

if FINAL_VIDEO and os.path.exists(FINAL_VIDEO):
    size_mb = os.path.getsize(FINAL_VIDEO) / (1024 * 1024)
    print(f'\n✅ Avatar video ready: {FINAL_VIDEO} ({size_mb:.1f} MB)')
    files.download(FINAL_VIDEO)
    print('✅ Downloading...')
else:
    print('❌ No output video found — check error above')
    !ls -la /content/output/

In [ ]:
# ============================================================
# CELL 6 — Generate new video with custom text (reuse assets)
# ============================================================

# ✏️ Change only this text
CUSTOM_TEXT = "Good morning everyone. Today I will walk you through our AI avatar project. Our system uses voice cloning and realistic face animation to create a convincing digital presenter."

import os, shutil, subprocess
from google.colab import files

VOICE_SAMPLE = '/content/voice_sample.wav'
CUSTOM_AUDIO = '/content/custom_voice.wav'
OUTPUT_DIR   = '/content/output'

# Find face photo
FACE_PHOTO = None
for ext in ['.jpg', '.jpeg', '.png']:
    if os.path.exists(f'/content/face{ext}'):
        FACE_PHOTO = f'/content/face{ext}'
        break

assert os.path.exists(VOICE_SAMPLE), '❌ Run Cell 4 first'
assert FACE_PHOTO, '❌ Run Cell 5 first (to upload photo)'

# Step 1: Generate custom voice
print(f'[1/2] Generating voice: "{CUSTOM_TEXT[:60]}..."')
subprocess.run([
    'f5-tts_infer-cli',
    '--model',       'F5TTS_v1_Base',
    '--ref_audio',   VOICE_SAMPLE,
    '--ref_text',    '',
    '--gen_text',    CUSTOM_TEXT,
    '--output_dir',  '/content/',
    '--output_file', 'custom_voice.wav',
    '--remove_silence',
], capture_output=True)

if not os.path.exists(CUSTOM_AUDIO):
    wavs = sorted([f for f in os.listdir('/content') if 'custom' in f and f.endswith('.wav')])
    if wavs:
        shutil.copy(f'/content/{wavs[-1]}', CUSTOM_AUDIO)

print(f'✅ Audio: {CUSTOM_AUDIO}')

# Step 2: SadTalker
print('\n[2/2] Generating realistic avatar...')
%cd /content/SadTalker
!python inference.py \
    --driven_audio  "{CUSTOM_AUDIO}" \
    --source_image  "{FACE_PHOTO}" \
    --result_dir    "{OUTPUT_DIR}" \
    --still \
    --preprocess    full \
    --enhancer      gfpgan \
    --size          256

# Find latest output
CUSTOM_VIDEO = None
for f in sorted(os.listdir(OUTPUT_DIR), reverse=True):
    if f.endswith('.mp4'):
        CUSTOM_VIDEO = os.path.join(OUTPUT_DIR, f)
        break

if CUSTOM_VIDEO and os.path.exists(CUSTOM_VIDEO):
    print(f'\n✅ Done! Video: {CUSTOM_VIDEO}')
    files.download(CUSTOM_VIDEO)
else:
    print('❌ Failed — check errors above')